# MLP search on Kaggle
Optuna TPE + Hyperband, 100 trials (`conf/hyperparameters/search.yaml`), about 6 hours on a T4. The study is saved in
`study.log`, so a search cut by Kaggle's 12-hour limit continues in the next session.
- Settings: GPU, Internet on. Input: the dataset `sandrokhizanishvili/hi-small-gnn`. Run with Save Version -> Save & Run All.
- Search: `N_TRIALS = 100`.
- Next session (search cut short, or the final after the search is checked): add the previous output as input
  (Add Input -> Notebook Output Files), set `PREVIOUS`; for the final also `RUN_FINAL = True`.
- Output: `/kaggle/working/outputs/tuning/mlp/mlp_readout-full/` (`study.log`, `trials.csv`, `tuning_curves.png`, `recipe.json`).

In [ ]:
N_TRIALS = 100       # the session stops when the study has this many trials
PREVIOUS = ''        # the previous session's outputs folder, '' in the first session
RUN_FINAL = False    # True only after the search is checked

In [ ]:
# Pinned torch / PyG stack for the Kaggle GPU.
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric lightgbm==4.6.0 hydra-core==1.3.2 optuna==5.0.0

In [ ]:
# The code from GitHub, then the previous session's study (recipe.json is written again at the end).
import os
if os.path.isdir('/kaggle/working/gnn_for_aml'):
    !cd /kaggle/working/gnn_for_aml && git pull
else:
    !git clone https://github.com/sandrokhizanishvili/gnn_for_aml.git /kaggle/working/gnn_for_aml
%cd /kaggle/working/gnn_for_aml
!ls /kaggle/input
if PREVIOUS:
    !mkdir -p /kaggle/working/outputs && cp -rn {PREVIOUS}/tuning /kaggle/working/outputs/ && rm -f /kaggle/working/outputs/tuning/*/*/recipe.json && ls /kaggle/working/outputs/tuning/*/*

In [ ]:
# Environment check.
!python run.py mode=check machine=kaggle

In [ ]:
# The search; trials already in study.log are kept.
!python run.py tuning=mlp/search hyperparameters.search.n_trials={N_TRIALS} machine=kaggle

In [ ]:
# Trials so far, best first.
import glob, optuna, pandas as pd
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend, JournalFileOpenLock
path = glob.glob('/kaggle/working/outputs/tuning/mlp/*/study.log')[0]
study = optuna.load_study(study_name=os.path.basename(os.path.dirname(path)),
                          storage=JournalStorage(JournalFileBackend(path, lock_obj=JournalFileOpenLock(path))))
trials = study.trials_dataframe()
print(trials['state'].value_counts().to_dict())
trials.sort_values('value', ascending=False).head(15)

In [ ]:
# Final: the winner refit on train + val, seeds 42-46; a finished seed is skipped.
if RUN_FINAL and glob.glob('/kaggle/working/outputs/tuning/mlp/*/recipe.json'):
    !python run.py -m tuning=mlp/final machine=kaggle

In [ ]:
# Small archive for download (no checkpoints, no predictions).
!cd /kaggle/working && rm -f results_small.zip && zip -qr results_small.zip outputs -x "*.pt" "*predictions.csv" && ls -lh results_small.zip

## After the run
- Download `results_small.zip` from the Output tab and unzip it into `gnn_for_aml/outputs/`.
- Fewer than 100 trials: next session with `PREVIOUS` set.